# Pharma Product Matcher (Finland, Kela reimbursement data)

Match a messy medicine name (e.g. `burana 400mg tabl 30`) to the exact product in the official Kela list,
using **multilingual embeddings + blocking + business rules + an LLM second opinion for uncertain cases**,
and measure how well it works.

**Why it matters for pricing:** before comparing prices across sources or countries, you must know it is the *same* product.
A wrong match puts a wrong price into the analysis.

**Pipeline**
```
messy name → normalise → embed → block by active substance → rank by cosine similarity
          → strength rule → pack rule → name rule → threshold → match / not sure / no match
          → "not sure" only → LLM judge (Gemini, JSON verdict, validated) → final decision
```

**Results on 285 unseen products from 22 medicines:**

| Version | Precision | Recall |
|---|---|---|
| v1 (held-out: system frozen before the run, evaluated once) | 96% | 88% |
| v1.1 (strength-parser fixes + LLM judge)* | 99.6% | 99.6% |

\* v1.1 was designed by studying v1's errors on this set, so the set is no longer truly held-out
and these numbers are optimistic. A fresh test set is the next step.

Data: public Kela medicine price data (kela.fi). Personal project; no employer data or tools used.

## 1. Setup

In [1]:
!pip install -q sentence-transformers

In [2]:
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## 2. Load and clean the Kela master list

Data-quality issues found and fixed:
- **VNR lost its leading zeros** in Excel → read as text, `zfill(6)`
- **Product type mixed into the name** ("self-care / prescription medicine") → moved to an `Rx` column
- **Invisible zero-width space `\u200b`** inside text (found with `repr()`) → removed
- **Prices ×100** from decimal-comma parsing → validated against Kela and fixed

In [3]:
df = pd.read_excel("finland_KELA.xlsx", dtype={"VNR": str})
df["VNR"] = df["VNR"].str.zfill(6)

df["Rx"] = df["Product"].str.contains("prescription")
df["Product"] = (df["Product"]
                 .str.replace("self-care medicine", "")
                 .str.replace("prescription medicine", "")
                 .str.strip())

for col in ["Product", "Strength", "Dosage Form", "Package Size"]:
    df[col] = df[col].str.replace("\u200b", "")

df = df.reset_index(drop=True)


df["full_name"]  = df["Product"] + " " + df["Strength"] + " " + df["Dosage Form"] + " " + df["Package Size"]
df["embed_name"] = df["Product"] + " " + df["Strength"] + " " + df["Dosage Form"]

print(df.shape)
print(df["Search Product"].value_counts())
print("price range (€):", df["Price"].min(), "-", df["Price"].max())   # sanity check

(344, 16)
Search Product
amlodipin       46
pregabalin      38
burana          30
losartan        29
atorvastatin    29
simvastatin     26
panadol         20
sertralin       18
amorion         14
metformin       13
somac           11
omeprazol       10
seretide         9
pamol            9
zyrtec           7
nexium           7
ventoline        6
nucala           4
primaspan        4
aerius           3
havrix           3
blenrep          2
thyroxin         2
marevan          2
dovato           1
avamys           1
Name: count, dtype: int64
price range (€): 1.55 - 22171.45


## 3. Embed the master list

`paraphrase-multilingual-MiniLM-L12-v2`: 384-dimensional vectors, understands Finnish.
(The English-only `all-MiniLM-L6-v2` was tried first and replaced because the data is Finnish.)

In [4]:
model_ml = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kela_vectors = model_ml.encode(df["embed_name"].tolist())
print(kela_vectors.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(344, 384)


## 4. Helper functions

- `cosine`: similarity between two vectors (1 = same meaning)
- `strength`: the full strength with its unit, in a fixed format (`"400 mg"` → `"400 mg"`, `"5/160 mg"` → `"5/160 mg"`,
  `"100/12,5 mg"` → `"100/12,5 mg"`, `"27,5 mikrog/suihke"` → `"27,5 mikrog"`, `"1440 ELISA U/ml"` → `"1440 elisa u"`).
  Units: mg, mikrog, g, elisa u. Number and unit are returned together, so 25 mg ≠ 25 mikrog.
- `package_size`: the first number in Kela's pack text (`"30 fol ..."` → `"30"`)
- `query_pack`: the pack number in a query. It removes the strength text first (same regex as `strength`), then takes
  the first remaining number (`"... 5/160mg tabl 100"` → `"100"`; `"buranagel 50mg/g 50 g"` → `"50"`)
- `normalise`: lowercase, `400mg` → `400 mg`, abbreviations → Kela words
- `name_in_query`: every word of a product name appears in the query (hyphen-tolerant, checks raw and normalised query)

In [5]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

STRENGTH_RE = r"(\d+(?:[.,]\d+)?(?:/\d+(?:[.,]\d+)?)*)\s*(mikrog|mg|g|elisa u)"

def strength(text):
    m = re.search(STRENGTH_RE, text, flags=re.IGNORECASE)
    if m is None:
     return None
    return f"{m.group(1)} {m.group(2).lower()}"

def package_size(text):
    m = re.search(r"(\d+)", text)
    if m is None:
        return None
    return m.group(1)

def query_pack(q):
    q = re.sub(STRENGTH_RE, "", q, count=1, flags=re.IGNORECASE)
    nums = re.findall(r"\d+", q)
    if len(nums) == 0:
        return None
    return nums[0]

ABBREV = {"tabl": "tabletti", "tab": "tabletti", "caps": "kapseli", "kaps": "kapseli"}

def normalise(q):
    q = q.lower()
    q = re.sub(r"(\d+)\s*mg", r"\1 mg", q)
    words = q.split()
    words = [ABBREV.get(w, w) for w in words]
    return " ".join(words)

def name_in_query(name, query):
    raw  = query.lower().replace("-", " ").split()
    norm = normalise(query).replace("-", " ").split()
    q_words = set(raw) | set(norm)
    for w in name.lower().replace("-", " ").split():
        if w not in q_words:
            return False
    return True

print(strength("AVAMYS 27,5 mikrog/suihke"), "|", query_pack("avamys 27,5mikrog/suihke nenäsumute 120"))
print(strength("burana 400mg tabl 30"),      "|", query_pack("burana 400mg tabl 30"))
print(strength("x 5/160 mg"),                "|", query_pack("amlodipin/valsartan krka 5/160mg tabl 100"))

27,5 mikrog | 120
400 mg | 30
5/160 mg | 100


## 5. Retrieval: `search()` = blocking + ranking

Keeps only products with the same active substance (**blocking**), scores each with cosine similarity, sorts.
Stays broad on purpose, so it can be reused (e.g. for a RAG Q&A layer later).

In [6]:
def search(query, substance, k=10):
    q = model_ml.encode(query)
    block = df[df["Active Substance"] == substance].copy()
    block["score"] = [cosine(q, kela_vectors[i]) for i in block.index]
    return (block.sort_values("score", ascending=False)
                 .head(k)[["Product", "full_name", "Strength", "Package Size", "score"]])

search(normalise("burana 400mg tabl"), "ibuprofeeni", 5)

,Product,full_name,Strength,Package Size,score
28,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 100...",400 mg,"100 kpl tablettipurkki, muovi",0.913135
29,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 20 ...",400 mg,20 fol läpipainopakkaus,0.913135
26,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 10 ...",400 mg,10 fol läpipainopakkaus,0.913135
27,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 10 ...",400 mg,"10 kpl tablettipurkki, muovi",0.913135
30,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 30 ...",400 mg,30 fol läpipainopakkaus,0.913135


## 6. Matching: `match()` = retrieval + business rules + thresholds

1. normalise the query and retrieve the **whole substance block** (a top-10 cut-off once dropped the right answer: *retrieval recall*)
2. **strength rule**: same strength **and unit**, including every number in a combination (5/160 mg ≠ 10/160 mg; 25 mg ≠ 25 mikrog)
3. **pack rule**: same pack count (a 10-pack and a 100-pack have different prices)
4. **name rule**: product name must appear in the query; prefer the most specific name (PANADOL ZAPP over PANADOL)
5. **thresholds**: ≥ 0.75 match · < 0.60 no match · otherwise **not sure** → sent to the LLM judge (Section 13)

In [7]:
MATCH_T, NO_MATCH_T = 0.75, 0.60

def match(query, substance):
    q = normalise(query)
    topCandidates = search(q, substance, 1000)

    q_mg = strength(q)
    if q_mg is not None:
        topCandidates = topCandidates[topCandidates["Strength"].apply(strength) == q_mg]

    q_pack = query_pack(q)
    if q_pack is not None:
        topCandidates = topCandidates[topCandidates["Package Size"].apply(package_size) == q_pack]

    if len(topCandidates) == 0:
        return ("no match", None, 0.0)

    keepPdt = [name_in_query(n, query) for n in topCandidates["Product"]]
    topCandidates = topCandidates[keepPdt].copy()
    if len(topCandidates) == 0:
        return ("no match", None, 0.0)

    topCandidates["name_len"] = topCandidates["Product"].str.split().str.len()
    topCandidates = topCandidates.sort_values(["name_len", "score"], ascending=[False, False])

    top = topCandidates.iloc[0]
    score = float(top["score"])
    if score >= MATCH_T:
        decision = "match"
    elif score < NO_MATCH_T:
        decision = "no match"
    else:
        decision = "not sure"
    return (decision, top["full_name"], round(score, 3))

## 7. Regression tests (re-run after every change)

Every fix can silently break something that used to work, so these checks are re-run after each change.

- **Matcher cases:** known queries and what `match()` should return (strength variants, a strength that doesn't exist,
  the BURANA-CAPS capsule, pack sizes). This caught the name rule breaking BURANA-CAPS in v1.
- **Parser tests (v1.1):** `assert` checks for `strength()` and `query_pack()`. If any fails, the cell stops with an error,
  so a broken parser can't go unnoticed.

In [8]:
for q in ["burana 400mg tabl", "burana 200mg tabl", "burana 600mg tabl",
          "burana 300mg tabl", "burana caps 400mg", "burana 400mg tabl 30",
          "burana 400mg tabl 100", "burana 400mg tabl 999"]:
    print(q, "→", match(q, "ibuprofeeni"))

burana 400mg tabl → ('match', 'BURANA 400 mg tabletti, kalvopäällysteinen 100 kpl tablettipurkki, muovi', 0.913)
burana 200mg tabl → ('match', 'BURANA 200 mg tabletti, kalvopäällysteinen 30 fol läpipainopakkaus', 0.913)
burana 600mg tabl → ('match', 'BURANA 600 mg tabletti, kalvopäällysteinen 30 fol läpipainopakkaus', 0.917)
burana 300mg tabl → ('no match', None, 0.0)
burana caps 400mg → ('match', 'BURANA-CAPS 400 mg kapseli, pehmeä 100 fol läpipainopakkaus', 0.815)
burana 400mg tabl 30 → ('match', 'BURANA 400 mg tabletti, kalvopäällysteinen 30 fol läpipainopakkaus', 0.902)
burana 400mg tabl 100 → ('match', 'BURANA 400 mg tabletti, kalvopäällysteinen 100 kpl tablettipurkki, muovi', 0.866)
burana 400mg tabl 999 → ('no match', None, 0.0)


In [9]:
# v1.1 parser tests: the cell fails loudly if any of these breaks
assert strength("burana 400mg tabl") == "400 mg"
assert strength("5/160 mg") == "5/160 mg"
assert strength("100/12,5 mg") == "100/12,5 mg"
assert strength("27,5 mikrog/suihke") == "27,5 mikrog"
assert strength("1440 ELISA U/ml") == "1440 elisa u"
assert strength("no strength here") is None

assert query_pack("burana 400mg tabl 30") == "30"
assert query_pack("amlodipin/valsartan krka 5/160mg tabl 100") == "100"
assert query_pack("buranagel 50mg/g 50 g") == "50"
assert query_pack("avamys 27,5mikrog nenäsumute 120") == "120"

print("all parser tests passed")

all parser tests passed


## 8. Test set with known answers

`make_messy()` turns each Kela row into a messy query, so the correct answer is known (synthetic ground truth).
Limitation: generated queries are tidier than real-world mess.

**v1.1 fix:** `make_messy()` now keeps the full strength and unit (`5/160mg`, `27,5mikrog`). In v1 it kept only the
number before "mg", so 5/160 and 10/160 products produced the **same** query. That was a test-generator bug,
not just a matcher bug.

In [10]:
SHORT = {"tabletti": "tabl", "kapseli": "kaps"}

def make_messy(row):
    brand = row["Product"].lower()
    mg = strength(row["Strength"])
    s = mg.replace(" ", "", 1) if mg is not None else row["Strength"].lower()
    form = row["Dosage Form"].split(",")[0]
    form = SHORT.get(form, form)
    pack = package_size(row["Package Size"])
    return f"{brand} {s} {form} {pack}"

tests = pd.DataFrame({
    "query":     df.apply(make_messy, axis=1),
    "substance": df["Active Substance"],
    "answer":    df["full_name"],
})
tests.head()

,query,substance,answer
0,"avamys 27,5mikrog nenäsumute 120",flutikasonifuroaatti,"AVAMYS 27,5 mikrog/suihke nenäsumute, suspensi..."
1,seretide diskus 50/100mikrog inhalaatiojauhe 3,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/100 mikrog/annos inhalaatio...
2,seretide diskus 50/100mikrog inhalaatiojauhe 60,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/100 mikrog/annos inhalaatio...
3,seretide diskus 50/250mikrog inhalaatiojauhe 3,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/250 mikrog/annos inhalaatio...
4,seretide diskus 50/250mikrog inhalaatiojauhe 60,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/250 mikrog/annos inhalaatio...


## 9. Run the matcher on every test query and mark it

"Correct" = same product, strength, form and **pack count** (blister vs bottle of the same count had identical prices in the data).

In [11]:
decisions, predicted, scores = [], [], []
for q, s in zip(tests["query"], tests["substance"]):
    d, p, sc = match(q, s)
    decisions.append(d); predicted.append(p); scores.append(sc)
tests["decision"], tests["predicted"], tests["score"] = decisions, predicted, scores

df["key"] = df["embed_name"] + " | " + df["Package Size"].apply(package_size)
key_of = dict(zip(df["full_name"], df["key"]))

correct_list = []
for p, a in zip(tests["predicted"], tests["answer"]):
    if p is None:
        correct_list.append(False)
    else:
        correct_list.append(key_of[p] == key_of[a])
tests["correct"] = correct_list

## 10. Results: dev set vs unseen test set

**v1 (recorded result):** the system was **frozen** (rules + threshold 0.75) before the unseen products were scraped,
and evaluated on them once: **96% precision, 88% recall.**

**The cells below now run v1.1.** v1.1 was designed by studying v1's errors on this unseen set, so the set is
no longer truly held-out and the v1.1 numbers below are optimistic. A fresh test set is the next step.

In [12]:
DEV = ["burana", "panadol", "pamol"]
is_new = ~df["Search Product"].str.lower().isin(DEV)
unseen, dev = tests[is_new], tests[~is_new]

def report(t, name, col):
    m = t[t[col] == "match"]
    print(f"--- {name}: {len(t)} queries ---")
    print(t[col].value_counts().to_string())
    print(f"Precision: {m['correct'].mean():.1%}")
    print(f"Recall:    {m['correct'].sum() / len(t):.1%}")
    print()

report(dev, "DEV (seen)", "decision")
report(unseen, "UNSEEN (new products)", "decision")

--- DEV (seen): 59 queries ---
decision
match       57
not sure     2
Precision: 100.0%
Recall:    96.6%

--- UNSEEN (new products): 285 queries ---
decision
match       271
not sure     14
Precision: 99.6%
Recall:    94.7%



## 11. Error analysis of v1 (unseen)

The 11 wrong matches on unseen products in v1 fell into two patterns:
- **10 combination strengths** (`5/160 mg` vs `10/160 mg`; `100/12,5 mg` vs `50/12,5 mg`) → the strength regex read only one number
  (and split decimals at the comma), so the rule couldn't separate variants
- **1 device variant** (Nucala pre-filled pen vs pre-filled syringe, same strength and pack)

None of the errors came from retrieval, blocking, the pack rule or the name rule.

These findings led to the v1.1 fixes (Section 12b). Running the cell below now shows the **1 remaining** error (Nucala).

In [13]:
bad = unseen[(unseen["decision"] == "match") & (unseen["correct"] == False)]
print(len(bad), "wrong matches on unseen")
bad[["query", "answer", "predicted", "score"]]

1 wrong matches on unseen


,query,answer,predicted,score
11,nucala biologic 100mg injektioneste 1,"NUCALA Biologic 100 mg 1 ml, kiinteä neula, tu...","NUCALA Biologic 100 mg 1 ml, kiinteä neula inj...",0.796


## 12. Version history

**Dev set (59 products):**

| Version | Change | Precision | Recall |
|---|---|---|---|
| v1 | first run | 88% | 65% |
| v2 | pack bug fix + full product names in tests | 92% | 67% |
| v3 | name rule (brand / variant) | 97% | 67% |
| v4 | retrieve the whole substance block (retrieval recall) | 100% | 67% |
| v4 @ 0.75 | threshold chosen before the unseen run | 100% | 97% |

**Unseen set (285 products, 22 medicines):**

| Version | Change | Precision | Recall |
|---|---|---|---|
| v1 | frozen, evaluated once (held-out) | 96% | 88% |
| v1.1a* | combination strengths | 99.6% | 92.3% |
| v1.1b* | units (mikrog, g, elisa u) | 99.6% | 94.7% |
| v1.1c* | LLM judge on the 14 "not sure" cases | 99.6% | 99.6% |

\* designed using this set's errors, so optimistic.

## 12b. v1.1: strength parser fixes (from the error analysis)

### Combination strengths
- v1: 10 of 11 wrong matches were combination strengths (e.g. 5/160 vs 10/160).
  Cause: strength() read only the number before "mg" ("160"), in both the matcher and the test generator.
- Fix: strength() reads full combination strengths ("5/160") and the unit; query_pack() updated to match.
- After the fix: 0 wrong combination matches. Remaining error: 1 (Nucala pen vs syringe; the query doesn't say which device).

### Units
- Exploration showed 16 "no match" cases were all non-mg units (mikrog, g, elisa u).
- After the fix: 0 "no match"; 14 "not sure" cases, all actually correct → passed to the LLM judge (Section 13).

## 13. v1.1: LLM second opinion (Gemini)

Only the **"not sure"** cases (score 0.60–0.75) go to an LLM judge; everything else is decided by the rules.
That's 16 calls instead of 344, which matters for cost, speed and the free-tier quota (20 calls/day per model).

Model: `gemini-3.5-flash-lite`, pinned so results are reproducible. Temperature 0, JSON output, validated before use.
If the answer is missing or invalid, the case stays "not sure"; the system never guesses.

### 13.1 Connect to Gemini

In [14]:
RUN_LLM = False     # True = call Gemini (needs GEMINI_API_KEY in Secrets); False = use cached results

!pip install -q google-genai
from google import genai

if RUN_LLM:
    from google.colab import userdata
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# for m in client.models.list():
#     print(m.name)


### 13.2 ask_gemini(): pinned model, temperature 0, JSON mode, retries

In [15]:
import time
from google.genai import errors, types

MODEL = "gemini-3.5-flash-lite"     # pinned for reproducible evaluation (see README)

JUDGE_CONFIG = types.GenerateContentConfig(temperature=0,response_mime_type="application/json")

def ask_gemini(prompt, model=MODEL, tries=4):
    for attempt in range(tries):
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=JUDGE_CONFIG)
            return response.text
        except errors.ServerError as e:          # only "server busy / failed" errors are retried
            wait = 2 ** attempt
            print(f"Attempt {attempt+1}: server error {e.code}, retrying in {wait}s")
            time.sleep(wait)
    return None

# print(ask_gemini("Explain how AI works in a few words"))

### 13.3 build_prompt(): role, inputs, rules, output format

In [16]:
def build_prompt(query, candidate):
  prompt = f"""You are a pharma product description checker. Decide whether the query and the candidate describe the same product from the Finnish Kela medicine list.
  Query: {query}
  Candidate: {candidate}

  Rules:
  1. Product name must match, including variant words (e.g. ZAPP, CAPS, COMP make it a different product).
  2. Strength must match exactly, including every number in a combination
    (5/160 mg is not 10/160 mg) and the unit (mg is not mikrog).
  3. Pack count must match. Blister and bottle packs with the same count are the same product.
  4. The query may shorten words (tabl = tabletti, kaps = kapseli) or leave out details
    such as "kalvopäällysteinen". Missing details are NOT a mismatch; only a contradiction is.
  5. The data is Finnish: tabletti = tablet, kapseli = capsule, mikrog = micrograms,
    injektioneste = solution for injection.

  Reply only with JSON, no other text, in this format:
  {{"match": true or false, "confidence": a number from 0 to 1 , "reason":"one short sentence"}}"""
  return prompt

In [17]:
p = build_prompt("simvastatin krka 10mg tabl 30",
                   "SIMVASTATIN KRKA 10 mg tabletti, kalvopäällysteinen 30 fol")
print(p)
# print(ask_gemini(p))

You are a pharma product description checker. Decide whether the query and the candidate describe the same product from the Finnish Kela medicine list.
  Query: simvastatin krka 10mg tabl 30
  Candidate: SIMVASTATIN KRKA 10 mg tabletti, kalvopäällysteinen 30 fol

  Rules:
  1. Product name must match, including variant words (e.g. ZAPP, CAPS, COMP make it a different product).
  2. Strength must match exactly, including every number in a combination
    (5/160 mg is not 10/160 mg) and the unit (mg is not mikrog).
  3. Pack count must match. Blister and bottle packs with the same count are the same product.
  4. The query may shorten words (tabl = tabletti, kaps = kapseli) or leave out details
    such as "kalvopäällysteinen". Missing details are NOT a mismatch; only a contradiction is.
  5. The data is Finnish: tabletti = tablet, kapseli = capsule, mikrog = micrograms,
    injektioneste = solution for injection.

  Reply only with JSON, no other text, in this format:
  {"match": true o

### 13.4 parse_verdict(): never trust the LLM's output unchecked

In [18]:
import json
def parse_verdict(answer):
  if answer is None:
    return None
  try:
    d = json.loads(answer)
  except json.JSONDecodeError:
    return None

  d_match = d.get("match")
  if not isinstance(d_match, bool):
    return None
  d_confidence = d.get("confidence")
  if not isinstance(d_confidence, (int, float)) or not (0 <= d_confidence <= 1):
    return None
  d_reason = d.get("reason")

  return d_match, d_confidence, d_reason


In [19]:
print(parse_verdict('{"match": true, "confidence": 0.9, "reason": "same"}'))
print(parse_verdict(None))
print(parse_verdict('Sure! {"match": true'))
print(parse_verdict('{"match": "yes", "confidence": 0.9, "reason": "x"}'))
print(parse_verdict('{"match": true, "confidence": 7, "reason": "x"}'))

(True, 0.9, 'same')
None
None
None
None


### 13.5 Columns for the LLM results

In [20]:
tests["final"] = tests["decision"]
tests["llm_reason"] = ""
tests["llm_done"] = False
unsure_final = tests[tests["decision"] == "not sure"]
print(len(unsure_final))

16


### 13.6 decide_final(): turning a verdict into a decision

In [21]:
LLM_T = 0.8

def decide_final(verdict):
    if verdict is None:
        return "not sure", ""
    match, conf, reason = verdict
    if match and conf >= LLM_T:
        return "match", reason
    if not match:
        return "no match", reason
    return "not sure", reason

print(decide_final(None))
print(decide_final((True, 0.95, "same product")))
print(decide_final((False, 0.9, "different pack")))
print(decide_final((True, 0.6, "unsure on form")))

('not sure', '')
('match', 'same product')
('no match', 'different pack')
('not sure', 'unsure on form')


### 13.7 Run the judge (resumable: skips done rows, stops cleanly on quota errors)

In [22]:
if RUN_LLM:
    for idx, row in unsure_final.iterrows():
        if tests.loc[idx, "llm_done"]:        # already judged -> skip, no API call
            continue
        try:
            llm_answer = ask_gemini(build_prompt(row["query"], row["predicted"]))
        except errors.ClientError as e:
            print("Stopped:", e.code, "- re-run later to continue")
            break
        verdict = parse_verdict(llm_answer)
        final, reason = decide_final(verdict)
        tests.loc[idx, ["final", "llm_reason", "llm_done"]] = [final, reason, True]
        print(idx, row["query"], "→", verdict, "→", final)
        time.sleep(2)

    tests.to_csv("tests_with_llm.csv", index=False)
    print(tests["llm_done"].sum(), "of", len(unsure_final), "done")

else:
    saved = pd.read_csv("tests_with_llm.csv")
    for col in ["final", "llm_reason", "llm_done"]:
        tests[col] = saved[col].values
    tests["llm_reason"] = tests["llm_reason"].fillna("")
    print("Loaded saved LLM results:", tests["llm_done"].sum(), "rows")

Loaded saved LLM results: 16 rows


## 14. Results after the LLM step

Recall on unseen: **94.7% → 99.6%**, with **no new wrong matches** (precision stays 99.6%; the 1 error is Nucala,
which never reached the LLM). Dev: 100% / 100%.

Caveats: (1) this set was used for error analysis; (2) the LLM has so far only seen **correct** candidates, so a
rejection test with deliberately wrong pairs is needed; (3) the queries are synthetic.

In [23]:
unseen, dev = tests[is_new], tests[~is_new]
report(dev, "DEV before LLM", "decision")
report(dev, "DEV after LLM", "final")
report(unseen, "UNSEEN before LLM", "decision")
report(unseen, "UNSEEN after LLM", "final")

--- DEV before LLM: 59 queries ---
decision
match       57
not sure     2
Precision: 100.0%
Recall:    96.6%

--- DEV after LLM: 59 queries ---
final
match    59
Precision: 100.0%
Recall:    100.0%

--- UNSEEN before LLM: 285 queries ---
decision
match       271
not sure     14
Precision: 99.6%
Recall:    94.7%

--- UNSEEN after LLM: 285 queries ---
final
match    285
Precision: 99.6%
Recall:    99.6%



## 15. Limitations and next steps

- **Rejection test:** check the LLM judge says "no" to deliberately wrong pairs
- **Fresh held-out test set** for honest v1.1 numbers
- **Device variants** (pen vs syringe) are not captured by the rules
- **Free-tier quota:** the batch is resumable and results are saved, so the LLM is never called twice for the same case
- **Next (v1.2):** RAG price Q&A on the same data, answering only from retrieved rows and citing VNR codes

   ## Appendix: blister vs bottle price check

In [24]:
# Extra check: does packaging type (blister vs bottle) change price for the same pack count?
df["pack"] = df["Package Size"].apply(package_size)
check = df.groupby(["Product", "Strength", "pack"])["Price"].agg(["count", "nunique", "min", "max"])
check[check["count"] > 1]

count  nunique    min    max
Product                 Strength pack                              
ATORVASTATIN RATIOPHARM 10 mg    100       2        2   2.27   2.77
                        20 mg    100       2        2   2.54   3.03
                        40 mg    100       2        2   4.09   4.57
BURANA                  400 mg   10        2        1   3.97   3.97
                                 30        2        1   9.99   9.99
PRIMASPAN               100 mg   100       2        1  11.76  11.76
SERTRALIN ORION         100 mg   100       2        1  19.76  19.76
                        50 mg    100       2        1  14.46  14.46
SOMAC                   40 mg    100       2        1   6.32   6.32
ZYRTEC                  10 mg    30        2        2  13.68  16.80
                        10 mg/ml 20        3        3  18.55  21.84